# Setup


In [ ]:
! pip install synthid-text[notebook]

from collections.abc import Sequence
import enum
import gc

import datasets
import huggingface_hub
from synthid_text import detector_mean
from synthid_text import logits_processing
from synthid_text import synthid_mixin
from synthid_text import detector_bayesian
import tensorflow as tf
import torch
import tqdm
import transformers
import immutabledict

# Model Selection


In [3]:


class ModelName(enum.Enum):
  GPT2 = 'gpt2'
  GEMMA_2B = 'google/gemma-2b-it'
  GEMMA_7B = 'google/gemma-7b-it'


model_name = 'google/gemma-7b-it' # @param ['gpt2', 'google/gemma-2b-it', 'google/gemma-7b-it']
MODEL_NAME = ModelName(model_name)

if MODEL_NAME is not ModelName.GPT2:
  huggingface_hub.notebook_login()


# Tokenizer

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME.value)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Configuration

In [6]:
DEVICE = (
    torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
)
DEVICE


DEFAULT_WATERMARKING_CONFIG = immutabledict.immutabledict({
    "ngram_len": 5,  # This corresponds to H=4 context window size in the paper.
    "keys": [
        654,
        400,
        836,
        123,
        340,
        443,
        597,
        160,
        57,
        29,
        590,
        639,
        13,
        715,
        468,
        990,
        966,
        226,
        324,
        585,
        118,
        504,
        421,
        521,
        129,
        669,
        732,
        225,
        90,
        960,
        654,
        400,
        836,
        123,
        340,
        443,
        597,
        160,
        57,
        29,
        590,
        639,
        13,
        715,
        468,
        990,
        966,
        226,
        324,
        585,
        118,
        504,
        421,
        521,
        129,
        669,
        732,
        225,
        90,
        960,
        654,
        400,
        836,
        123,
        340,
        443,
        597,
        160,
        57,
        29,
        590,
        639,
        13,
        715,
        468,
        990,
        966,
        226,
        324,
        585,
        118,
        504,
        421,
        521,
        129,
        669,
        732,
        225,
        90,
        960,
    ],
    "sampling_table_size": 2**16,
    "sampling_table_seed": 0,
    "context_history_size": 1024,
    "device": (
        torch.device("cuda:0")
        if torch.cuda.is_available()
        else torch.device("cpu")
    ),
})
CONFIG = DEFAULT_WATERMARKING_CONFIG

BATCH_SIZE = 8
NUM_BATCHES = 320
OUTPUTS_LEN = 1024
TEMPERATURE = 0.5
TOP_K = 40
TOP_P = 0.99
NUM_NEGATIVES = 10000
POS_BATCH_SIZE = 32
NUM_POS_BATCHES = 313
NEG_BATCH_SIZE = 32
# Truncate outputs to this length for training.
POS_TRUNCATION_LENGTH = 200
NEG_TRUNCATION_LENGTH = 200
# Pad trucated outputs to this length for equal shape across all batches.
MAX_PADDED_LENGTH = 1000
TEMPERATURE = 1.0

# Utility Functions

In [7]:
def load_model(
    model_name: ModelName,
    expected_device: torch.device,
    enable_watermarking: bool = False,
) -> transformers.PreTrainedModel:
  if model_name == ModelName.GPT2:
    model_cls = (
        synthid_mixin.SynthIDGPT2LMHeadModel
        if enable_watermarking
        else transformers.GPT2LMHeadModel
    )
    model = model_cls.from_pretrained(model_name.value, device_map='auto')
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
  else:
    model_cls = (
        synthid_mixin.SynthIDGemmaForCausalLM
        if enable_watermarking
        else transformers.GemmaForCausalLM
    )
    model = model_cls.from_pretrained(
        model_name.value,
        device_map='auto',
        torch_dtype=torch.bfloat16,
    )

  if str(model.device) != str(expected_device):
    raise ValueError('Model device not as expected.')

  return model


def _compute_perplexity(
    outputs: torch.LongTensor,
    scores: torch.FloatTensor,
    eos_token_mask: torch.LongTensor,
    watermarked: bool = False,
) -> float:
  """Compute perplexity given the model outputs and the logits."""
  len_offset = len(scores)
  if watermarked:
    nll_scores = scores
  else:
    nll_scores = [
        torch.gather(
            -torch.log(torch.nn.Softmax(dim=1)(sc)),
            1,
            outputs[:, -len_offset + idx, None],
        )
        for idx, sc in enumerate(scores)
    ]
  nll_sum = torch.nan_to_num(
      torch.squeeze(torch.stack(nll_scores, dim=1), dim=2)
      * eos_token_mask.long(),
      posinf=0,
  )
  nll_sum = nll_sum.sum(dim=1)
  nll_mean = nll_sum / eos_token_mask.sum(dim=1)
  return nll_mean.sum(dim=0)


def _process_raw_prompt(prompt: Sequence[str]) -> str:
  """Add chat template to the raw prompt."""
  if MODEL_NAME == ModelName.GPT2:
    return prompt.decode().strip('"')
  else:
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt.decode().strip('"')}],
        tokenize=False,
        add_generation_prompt=True,
    )

# @title Generate model responses and compute g-values

def generate_responses(example_inputs, enable_watermarking):
  inputs = tokenizer(
      example_inputs,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)

  # @title Watermarked output preparation for detector training
  gc.collect()
  torch.cuda.empty_cache()

  model = load_model(
      MODEL_NAME,
      expected_device=DEVICE,
      enable_watermarking=enable_watermarking,
  )
  torch.manual_seed(0)
  _, inputs_len = inputs['input_ids'].shape

  outputs = model.generate(
      **inputs,
      do_sample=True,
      max_length=inputs_len + OUTPUTS_LEN,
      temperature=TEMPERATURE,
      top_k=TOP_K,
      top_p=TOP_P,
  )

  outputs = outputs[:, inputs_len:]

  # eos mask is computed, skip first ngram_len - 1 tokens
  # eos_mask will be of shape [batch_size, output_len]
  eos_token_mask = logits_processor.compute_eos_token_mask(
      input_ids=outputs,
      eos_token_id=tokenizer.eos_token_id,
  )[:, CONFIG['ngram_len'] - 1 :]

  # context repetition mask is computed
  context_repetition_mask = logits_processor.compute_context_repetition_mask(
      input_ids=outputs,
  )
  # context repitition mask shape [batch_size, output_len - (ngram_len - 1)]

  combined_mask = context_repetition_mask * eos_token_mask

  g_values = logits_processor.compute_g_values(
      input_ids=outputs,
  )
  # g values shape [batch_size, output_len - (ngram_len - 1), depth]

  return g_values, combined_mask

import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

def find_best_threshold_at_fpr(y_true, y_scores, target_fpr=0.01):

    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)

    # Find the threshold where FPR is closest to target_fpr
    idx = np.argmin(np.abs(fpr - target_fpr))

    best_threshold = thresholds[idx]
    best_tpr = tpr[idx]
    actual_fpr = fpr[idx]

    return best_threshold, best_tpr, actual_fpr

# Load Dataset


In [ ]:
import json
import tensorflow_datasets as tfds
import datasets

ds = datasets.load_dataset("Pavithree/eli5")
id_to_prompt = {}
# Access the 'train' split of the dataset and iterate over it
for x in iter(ds['train'].to_iterable_dataset()):
    id_to_prompt[x['q_id']] = x['title']

full_data = []
with open('human_eval.jsonl') as f:
    for json_str in f:
        x = json.loads(json_str)
        # Check if the q_id exists in id_to_prompt before accessing it
        if x['q_id'] in id_to_prompt:
            x['question'] = id_to_prompt[x['q_id']]
            full_data.append(x)

## Mean Score TPR Trend


In [ ]:
import numpy as np
from tqdm import tqdm
from IPython.display import clear_output

wm_mean_scores = []
uwm_mean_scores = []
TPRs_mean = []
layers = []

for j in range(1, 100, 3):

    # Create a mutable copy of the CONFIG dictionary
    mutable_config = dict(CONFIG)
    mutable_config['keys'] = mutable_config['keys'][:j]
    logits_processor = logits_processing.SynthIDLogitsProcessor(
        **mutable_config, top_k=TOP_K, temperature=TEMPERATURE
    )
    for i in tqdm(range(50)):

        example_inputs = [full_data[i*20 + k]['question'] for k in range(20)]

        wm_g_values, wm_mask = generate_responses(
            example_inputs, enable_watermarking=True
        )
        uwm_g_values, uwm_mask = generate_responses(
            example_inputs, enable_watermarking=False
        )

        wm = detector_mean.mean_score(
            wm_g_values.cpu().numpy(), wm_mask.cpu().numpy()
        )

        uwm = detector_mean.mean_score(
            uwm_g_values.cpu().numpy(), uwm_mask.cpu().numpy()
        )

        wm_mean_scores = np.append(wm_mean_scores, wm)
        uwm_mean_scores = np.append(uwm_mean_scores, uwm)

        torch.cuda.empty_cache()
        clear_output(wait=True)

    values = np.concatenate([uwm_mean_scores, wm_mean_scores])
    labels = np.array([0] * len(uwm_mean_scores) + [1] * len(wm_mean_scores))

    threshold, tpr, actual_fpr = find_best_threshold_at_fpr(labels, values, target_fpr=0.01)
    TPRs_mean.append(tpr)
    layers.append(j)

# Attack

In [12]:
from collections import Counter
import numpy as np


def tournament_layer(tokens):
    unique_tokens = set(tokens)
    g_values = {x: np.random.binomial(1, 0.5) for x in unique_tokens}

    winners = []
    for i in range(0, len(tokens), 2):
        t1, t2 = tokens[i], tokens[i+1]
        if g_values[t1] > g_values[t2]:
            winners.append(t1)
        elif g_values[t2] > g_values[t1]:
            winners.append(t2)
        else:  # tie → random choice
            winners.append(np.random.choice([t1, t2]))
    return winners

def tournament_model(tokens, num_layers):
    for _ in range(num_layers):
        tokens = tournament_layer(tokens)
    return tokens


In [ ]:
model = load_model(MODEL_NAME, expected_device=DEVICE, enable_watermarking=True)
logits_processor = logits_processing.SynthIDLogitsProcessor(
    **CONFIG, top_k=TOP_K, temperature=TEMPERATURE)

In [ ]:
from scipy.special import softmax
import tqdm
from synthid_text import logits_processing
import torch

num_layer_att = 5
num_detect = 0

# Ensure model and config are loaded (assuming they are from previous cells)


# We iterate through a subset for demonstration to save time, or use the full range as requested
for i in range(10): # Reduced range for quick testing, change back to 1000 for full run
    output = ''
    # Using the prompt from the dataset
    prompt = full_data[i]['question']

    # Generate 100 new tokens
    for _ in tqdm.tqdm(range(100)):
        # Duplicate prompt to generate multiple candidates in parallel
        # 2^num_layer_att candidates for the tournament
        inputs = tokenizer([prompt] * (2 ** num_layer_att), return_tensors='pt', padding=True).to(DEVICE)

        # Generate 1 new token for each candidate
        next_token_ids = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.7,
            max_new_tokens=1, # Fix: Use max_new_tokens instead of max_length
            top_k=40,
            pad_token_id=tokenizer.eos_token_id
        )

        # Extract just the newly generated token IDs
        # next_token_ids shape: [batch_size, input_len + 1]
        new_tokens = next_token_ids[:, -1].tolist()

        # Run the tournament on the generated tokens
        winner_token_id = tournament_model(new_tokens, num_layer_att)

        # The result of tournament_model is a list containing the winner, or a single value depending on implementation.
        # Assuming tournament_model returns a list of winners (and finally 1 winner)
        if isinstance(winner_token_id, list):
             winner_token_id = winner_token_id[0]

        next_token_str = tokenizer.decode(winner_token_id, skip_special_tokens=True)

        if not next_token_str:
            break

        prompt += next_token_str
        output += next_token_str


    # Evaluation code needs to handle the final output
    eos_token_mask = logits_processor.compute_eos_token_mask(
        input_ids=tokenizer(output, return_tensors='pt', add_special_tokens=False)['input_ids'].to(DEVICE),
        eos_token_id=tokenizer.eos_token_id,
    )[:, CONFIG['ngram_len'] - 1 :]

    context_repetition_mask = logits_processor.compute_context_repetition_mask(
        input_ids=tokenizer(output, return_tensors='pt', add_special_tokens=False)['input_ids'].to(DEVICE)
    )

    combined_mask = context_repetition_mask * eos_token_mask

    # g_values computation requires input_ids
    g_values = logits_processor.compute_g_values(
        input_ids=tokenizer(output, return_tensors='pt', add_special_tokens=False)['input_ids'].to(DEVICE)
    )

    # Score
    score = detector_mean.mean_score(g_values.cpu().numpy(), combined_mask.cpu().numpy())

    # Assuming 'tau' is defined somewhere or we pick a threshold.
    # If tau is not defined, we can print the score.
    # if score > tau:
    #    num_detect += 1
    print(output)
    print(f"Example {i} Score: {score}")

# print(num_detect)

# Finding threshold

In [ ]:
wm_scores = []
uwm_scores = []

# Configure logits processor (re-initializing to be safe, using current config)
logits_processor = logits_processing.SynthIDLogitsProcessor(
    **CONFIG, top_k=TOP_K, temperature=TEMPERATURE
)

print("Generating responses and computing scores for 10 examples...")
for i in tqdm.tqdm(range(1000)):
    prompt = full_data[i]['question']

    # Generate watermarked response and score
    wm_g_values, wm_mask = generate_responses(
        [prompt], enable_watermarking=True
    )
    wm_score = detector_mean.mean_score(
        wm_g_values.cpu().numpy(), wm_mask.cpu().numpy()
    )
    wm_scores.append(wm_score)

    # Generate unwatermarked response and score
    uwm_g_values, uwm_mask = generate_responses(
        [prompt], enable_watermarking=False
    )
    uwm_score = detector_mean.mean_score(
        uwm_g_values.cpu().numpy(), uwm_mask.cpu().numpy()
    )
    uwm_scores.append(uwm_score)

print("Finished generating scores.")

In [ ]:
scores = np.concatenate([uwm_scores, wm_scores])
labels = np.array([0] * len(uwm_scores) + [1] * len(wm_scores))
find_best_threshold_at_fpr(y_true=labels, y_scores=scores,target_fpr=0.01)